<a href="https://colab.research.google.com/github/JakeOh/202511_BD53/blob/main/lab_ml/ml22_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXAONE 모델을 사용한 질문-답변 시스템

현재 Colab에 설치된 transformers 패키지의 버전은 5.0.0.

EXAONE 모델을 사용하기 위해서는 5.1.0 버전이 필요.

In [1]:
!pip install -U transformers==5.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 86.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Imports

In [2]:
import numpy as np
import scipy
import transformers

In [3]:
transformers.__version__

'5.1.0'

In [4]:
# 깃허브에 ipynb 파일을 업로드할 때 다운로드 상태(진행바) 표시줄 때문에 오류 발생
# -> 깃허브에 정상적으로 업로드되게 하기 위해서
transformers.utils.logging.disable_progress_bar()

# EXAONE-3.5 모델

In [6]:
pipe = transformers.pipeline(
    task='text-generation',
    model='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct',
    device=0,  # GPU 사용
    trust_remote_code=True
)

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [7]:
# 메시지 템플릿: 'role'과 'content'를 키로 갖는 dict들을 아이템으로 갖는 리스트.
# role: 역할(system, user, assistant)
# content: 내용
messages = [
    {
        'role': 'system',
        'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야. \
        확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는 \
        간단하고 친절한 답변을 생성해줘.'
    },
    {
        'role': 'user',
        'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'
    }
]

In [16]:
result = pipe(messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [17]:
print(result)
#> pipeline 호출 결과: 'generated_text' 키를 갖는 dict 1개를 저장하고 있는 리스트

[{'generated_text': [{'role': 'system', 'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'}, {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'}, {'role': 'assistant', 'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}]}]


In [18]:
len(result)

1

In [19]:
result[0]  #> dict

{'generated_text': [{'role': 'system',
   'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
  {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
  {'role': 'assistant',
   'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}]}

In [20]:
result[0].keys()

dict_keys(['generated_text'])

result - `[ { 'generated_text': [...] } ]`

In [21]:
result[0]['generated_text']
#> 'role'과 'content'를 키로 갖는 dict들의 list.
#> 3개의 dict: 첫 2개는 입력값, 마지막 1개 AI가 생성한 답변

[{'role': 'system',
  'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
 {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
 {'role': 'assistant',
  'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}]

In [22]:
result[0]['generated_text'][-1]

{'role': 'assistant',
 'content': '안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'}

In [23]:
result[0]['generated_text'][-1]['content']

'안녕하세요!   다이어리에 내년 공휴일이 포함되어 있는지 확인하는 건 정말 좋은 질문이에요! 😊\n\n제품 담당자께서 최대한 빨리 답변해드리도록 할게요. 곧 다시 연락드릴게요! 궁금한 점이 더 있으시면 언제든지 물어봐 주세요.  긍정적인 하루 보내세요! 👍'

pipeline은 실행할 때마다 다른 답변(텍스트)를 생성함.

pipeline의 파라미터:

*   `return_full_text`: 이전의 대화 기록을 모두 리턴할 지 여부를 설정. 기본값은 True.
*   `do_sample`: 샘플링 전략을 사용할 지 여부를 설정. 기본값은 True.
    *   `do_sample=True`: 메시지 프롬프트가 같아도 실행할 때마다 생성되는 텍스트가 달라짐.
    *   `do_sample=False`
        *   메시지 프롬프트가 같으면 항상 같은 텍스트가 생성됨.
        *   가장 확률이 높은 토큰들만 선택해서 텍스트를 생성.
*   샘플링 전략
    *   temperature(온도)
    *   top-k 방식
    *   top-p 방식


In [26]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              do_sample=False)
print(result)
#> result - [ { 'generated_text': '답변' } ]
#> 실행할 때마다 항상 같은 답변.

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요. 제품 담당자분께서 가장 정확한 정보를 제공해 드릴 수 있을 것 같아요. 그분께서는 다이어리의 최신 업데이트 내용과 내년 공휴일 정보를 확인하실 수 있으니, 조금 더 자세히 알아보시려면 연락을 취해보시는 건 어떨까요? 친절하게 답변해 드리겠습니다! 😊'}]


# 샘플링 전략 - temperature

In [28]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=10.0)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '친절해요 ^_^ 실제로 제품 사용 때 날짜 계산기와 더불에르로 내년 계획 세우셨었겠나요 🇨⟃ฃ Territories에 딱 들어가는지 자세히 살펴봐야 알터거예유!\\� 하지만 각제품은 해당국가 holidays(관 Holidays, 대개 국내 공식 정보에서 유래지만 꼭 출처 꼭check하시기 바랍니다!)까지 표시 해 두더고요 ✎정확인확인은 당사 서비스 매니저님이라 방문 시간이면 확인 부탁드려워 요✰그럼 그 시점 기준으로운율 말씀하시리때 더욱 편안할것으로보세효  함께 알아봐까요?! ✅\u200d♀⭐\u200d♀❤ 💗 💎 🚫 wait no~ 🤫 ơ느께 시간 되요 편하긴? 😻 📞 ✕ ✰⒌ 알려줘!! ✨ 🔽✔ ❤ ❤��\n- 쇼핑몰 제품 관리자를 대체 한 듯 쉽고도 세심하실답변들을 주실수 바래요.. 정확사항 확인 시에까지 최선응원해요 🤐☀�'}]


In [30]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=0.001)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요. 제품 담당자분께서 가장 정확한 정보를 제공해 드릴 수 있을 것 같아요. 그분께서는 다이어리의 최신 업데이트 내용과 내년 공휴일 정보를 확인하실 수 있으니, 조금 더 자세히 알아보시려면 연락을 취해보시는 건 어떨까요? 친절하게 답변해 드리겠습니다! 😊'}]


temperature(온도)를 1보다 큰 값을 사용하면 문장을 생성할 때 선택되는 토큰들이 다양해 지기 때문에 답변이 무작위짐.

temperature를 1보다 작은 값을 사용하면 높은 확률을 가진 토큰들이 선택될 가능성이 더 커짐. 생성되는 답변이 더 일관되어 짐.

In [31]:
# 동전 던지기
probs = [0.5, 0.5]  # 앞면 확률, 뒷면 확률

np.random.multinomial(n=100, pvals=probs)  # 동전 던지기 100번 실험

array([44, 56])

In [34]:
# 주사위 던지기
probs = [1/6] * 6
np.random.multinomial(n=600, pvals=probs)

array([ 98, 112, 105,  85, 107,  93])

logit(로짓): softmax 함수의 아규먼트. LLM에서는 어휘 사전에 포함된 각 토큰에 대한 점수.

In [35]:
logits = [1, 2, 3, 4, 10]  # 5개 토큰의 점수
probs = scipy.special.softmax(logits)
print(probs)

[1.22936559e-04 3.34176214e-04 9.08385131e-04 2.46924679e-03
 9.96165255e-01]


In [38]:
np.random.multinomial(n=200, pvals=probs)

array([  0,   0,   0,   1, 199])

In [42]:
logits = np.array([1, 2, 3, 4, 100])
probs = scipy.special.softmax(logits)
print(probs)

[1.01122149e-43 2.74878501e-43 7.47197234e-43 2.03109266e-42
 1.00000000e+00]


In [43]:
np.random.multinomial(n=200, pvals=probs)

array([  0,   0,   0,   0, 200])

In [44]:
probs = scipy.special.softmax(logits / 10)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[5.01629119e-05 5.54385914e-05 6.12691190e-05 6.77128484e-05
 9.99765417e-01]


array([  0,   0,   0,   0, 200])

In [45]:
probs = scipy.special.softmax(logits / 100)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[0.14810557 0.14959406 0.1510975  0.15261606 0.39858682]


array([29, 39, 33, 31, 68])

In [46]:
probs = scipy.special.softmax(logits / 0.1)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[0. 0. 0. 0. 1.]


array([  0,   0,   0,   0, 200])

*   logit 값들을 1보다 큰 수로 나눈 후 softmax 함수로 확률을 계산하면, logit 값이 작은 토큰들의 확률이 커지는 효과.
    *   토큰을 선택하는 다양성이 커짐.
    *   생성되는 텍스트가 다양해짐.
*   logit 값을 1보다 작은 수로 나눈 후 softmax 함수로 확률을 계산하면, logit 값이 큰 토큰들의 확률을 더 크게 만드는 효과
    *   토큰을 선택하는 다양성이 작아짐.
    *   거의 비슷한 텍스트들이 생성됨.
*   LLM에서 온도(temperature)는 logit 값들을 나눠주는 수.